# Chapter 1 Risk Trajectory Shapes

This notebook is a focused Chapter 1 review layer for baseline mortality-risk trajectories on ASIC.

It uses two distinct saved artifacts per model and horizon:

- `predictions.csv`: evaluation-only predictions on the horizon-labelable subset used for formal metrics.
- `all_valid_predictions.csv`: predictions for all scorable valid instances at that horizon, including unlabeled rows, for descriptive trajectory analysis.

Formal AUROC, AUPRC, calibration, and related metrics still belong to the labeled `predictions.csv` subset only. The all-valid export is for patient-course description, coverage, and trajectory-shape summaries.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "chapter1_mortality_decomposition").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current notebook session.")


REPO_ROOT = find_repo_root()
BASELINE_ROOT = REPO_ROOT / "artifacts" / "chapter1" / "baselines" / "asic" / "primary_medians"
HARD_CASE_ROOT = REPO_ROOT / "artifacts" / "chapter1" / "evaluation" / "asic" / "hard_cases" / "primary_medians"
MODEL_NAMES = ("logistic_regression", "xgboost")
HORIZONS = (8, 16, 24, 48, 72)
PRIMARY_MODEL = "logistic_regression"
PRIMARY_HORIZONS = (8, 24)

display(Markdown(f"**Repo root**: `{REPO_ROOT}`"))
display(Markdown(f"**Baseline artifact root**: `{BASELINE_ROOT}`"))


In [ ]:
def artifact_path(model_name: str, horizon_h: int, filename: str) -> Path:
    return BASELINE_ROOT / model_name / f"horizon_{int(horizon_h)}h" / filename


def load_prediction_export(model_name: str, horizon_h: int, filename: str) -> pd.DataFrame:
    path = artifact_path(model_name, horizon_h, filename)
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    for column in ("block_index", "prediction_time_h", "horizon_h", "block_start_h", "block_end_h"):
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")
    if "label_value" in df.columns:
        df["label_value"] = pd.to_numeric(df["label_value"], errors="coerce").astype("Int64")
    if "is_labelable" in df.columns:
        df["is_labelable"] = df["is_labelable"].astype("boolean")
    if "unlabeled_reason" in df.columns:
        df["unlabeled_reason"] = df["unlabeled_reason"].astype("string")
    df["stay_id_global"] = df["stay_id_global"].astype("string")
    df["hospital_id"] = df["hospital_id"].astype("string")
    df["model_name"] = model_name
    df["source_file"] = filename
    return df


def load_horizon_exports(model_name: str, horizon_h: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    evaluation = load_prediction_export(model_name, horizon_h, "predictions.csv")
    all_valid = load_prediction_export(model_name, horizon_h, "all_valid_predictions.csv")
    if "is_labelable" not in evaluation.columns:
        evaluation["is_labelable"] = True
    if "unlabeled_reason" not in evaluation.columns:
        evaluation["unlabeled_reason"] = pd.Series(pd.NA, index=evaluation.index, dtype="string")
    evaluation["prediction_scope"] = "evaluation_only"
    all_valid["prediction_scope"] = "all_valid"
    return evaluation, all_valid


def load_all_exports(model_names=MODEL_NAMES, horizons=HORIZONS) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    evaluation_frames = []
    all_valid_frames = []
    availability_rows = []
    for model_name in model_names:
        for horizon_h in horizons:
            try:
                evaluation, all_valid = load_horizon_exports(model_name, horizon_h)
            except FileNotFoundError:
                continue
            evaluation_frames.append(evaluation)
            all_valid_frames.append(all_valid)
            availability_rows.append(
                {
                    "model_name": model_name,
                    "horizon_h": horizon_h,
                    "evaluation_rows": int(evaluation.shape[0]),
                    "all_valid_rows": int(all_valid.shape[0]),
                    "additional_scored_rows": int(all_valid.shape[0] - evaluation.shape[0]),
                    "labelable_fraction_in_all_valid": float(all_valid["is_labelable"].fillna(False).mean()),
                }
            )
    if not evaluation_frames or not all_valid_frames:
        raise FileNotFoundError(
            "No Chapter 1 baseline prediction exports were found. Run the baseline models first."
        )
    return (
        pd.concat(evaluation_frames, ignore_index=True),
        pd.concat(all_valid_frames, ignore_index=True),
        pd.DataFrame(availability_rows).sort_values(["model_name", "horizon_h"]).reset_index(drop=True),
    )


def build_stay_coverage_table(all_valid_df: pd.DataFrame) -> pd.DataFrame:
    work = all_valid_df.copy()
    non_survivor_reason = work["unlabeled_reason"].astype("string").str.contains(
        "non_survivor",
        case=False,
        na=False,
    )
    work["positive_label"] = work["label_value"].eq(1)
    work["non_survivor_reason"] = non_survivor_reason
    coverage = work.groupby(["model_name", "horizon_h", "stay_id_global", "hospital_id"], as_index=False).agg(
        scored_blocks=("instance_id", "size"),
        evaluation_labeled_blocks=("is_labelable", lambda s: int(pd.Series(s).fillna(False).sum())),
        any_positive_label=("positive_label", "max"),
        any_non_survivor_reason=("non_survivor_reason", "max"),
    )
    coverage["additional_scored_unlabeled_blocks"] = (
        coverage["scored_blocks"] - coverage["evaluation_labeled_blocks"]
    )
    coverage["proxy_non_survivor"] = (
        coverage["any_positive_label"] | coverage["any_non_survivor_reason"]
    )
    return coverage


def compute_trajectory_features(all_valid_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    group_columns = ["model_name", "horizon_h", "stay_id_global", "hospital_id"]
    ordered_df = all_valid_df.sort_values(
        ["model_name", "horizon_h", "stay_id_global", "prediction_time_h", "block_index", "instance_id"],
        kind="stable",
    )
    for keys, stay_df in ordered_df.groupby(group_columns, dropna=False):
        stay_df = stay_df.reset_index(drop=True)
        probabilities = stay_df["predicted_probability"].astype(float)
        times = stay_df["prediction_time_h"].astype(float)
        if stay_df.empty:
            continue
        final_window = stay_df.loc[times.ge(times.max() - 24)].copy()
        if final_window.shape[0] < 2:
            final_window = stay_df.tail(min(3, stay_df.shape[0])).copy()
        if final_window.shape[0] >= 2 and final_window["prediction_time_h"].nunique(dropna=True) >= 2:
            slope = float(
                np.polyfit(
                    final_window["prediction_time_h"].astype(float),
                    final_window["predicted_probability"].astype(float),
                    1,
                )[0]
            )
        else:
            slope = np.nan
        upward_steps = probabilities.diff().gt(0)
        rows.append(
            {
                "model_name": keys[0],
                "horizon_h": int(keys[1]),
                "stay_id_global": keys[2],
                "hospital_id": keys[3],
                "n_scored_blocks": int(stay_df.shape[0]),
                "first_risk": float(probabilities.iloc[0]),
                "last_risk": float(probabilities.iloc[-1]),
                "max_risk": float(probabilities.max()),
                "mean_risk": float(probabilities.mean()),
                "change_first_to_last": float(probabilities.iloc[-1] - probabilities.iloc[0]),
                "final_window_slope": slope,
                "frac_blocks_ge_0_05": float(probabilities.ge(0.05).mean()),
                "frac_blocks_ge_0_10": float(probabilities.ge(0.10).mean()),
                "frac_blocks_ge_0_20": float(probabilities.ge(0.20).mean()),
                "max_occurs_in_final_block": bool(probabilities.idxmax() == len(probabilities) - 1),
                "upward_step_fraction": float(upward_steps.iloc[1:].mean()) if stay_df.shape[0] > 1 else np.nan,
            }
        )
    return pd.DataFrame(rows)


def load_hard_case_flags(model_name: str) -> pd.DataFrame | None:
    path = HARD_CASE_ROOT / model_name / "stay_level_hard_case_flags.csv"
    if not path.exists():
        return None
    hard_cases = pd.read_csv(path)
    hard_cases["stay_id_global"] = hard_cases["stay_id_global"].astype("string")
    hard_cases["horizon_h"] = pd.to_numeric(hard_cases["horizon_h"], errors="coerce").astype("Int64")
    return hard_cases


evaluation_predictions, all_valid_predictions, export_availability = load_all_exports()
stay_coverage = build_stay_coverage_table(all_valid_predictions)
trajectory_features = compute_trajectory_features(all_valid_predictions).merge(
    stay_coverage[
        [
            "model_name",
            "horizon_h",
            "stay_id_global",
            "hospital_id",
            "evaluation_labeled_blocks",
            "additional_scored_unlabeled_blocks",
            "proxy_non_survivor",
        ]
    ],
    on=["model_name", "horizon_h", "stay_id_global", "hospital_id"],
    how="left",
)
display(Markdown("## Available Exports"))
display(export_availability)


## 1. Sanity-Check Example Patient

Start with a concrete stay. If `asic_UK02_106` is present in the current artifact snapshot, use it. Otherwise fall back to the first stay that gains additional scored-but-unlabeled rows in the all-valid export.


In [ ]:
preferred_stay_id = "asic_UK02_106"
example_model = PRIMARY_MODEL
example_horizons = PRIMARY_HORIZONS

candidate_stay = None
available_stays = set(
    all_valid_predictions.loc[
        all_valid_predictions["model_name"].eq(example_model),
        "stay_id_global",
    ].astype("string")
)
if preferred_stay_id in available_stays:
    candidate_stay = preferred_stay_id
else:
    gained_rows = all_valid_predictions.merge(
        evaluation_predictions[["model_name", "horizon_h", "instance_id"]],
        on=["model_name", "horizon_h", "instance_id"],
        how="left",
        indicator=True,
    )
    gained_rows = gained_rows[gained_rows["_merge"].eq("left_only") & gained_rows["model_name"].eq(example_model)]
    if not gained_rows.empty:
        candidate_stay = gained_rows["stay_id_global"].astype("string").iloc[0]
    else:
        candidate_stay = all_valid_predictions.loc[
            all_valid_predictions["model_name"].eq(example_model),
            "stay_id_global",
        ].astype("string").iloc[0]

display(Markdown(f"### Example stay: `{candidate_stay}` ({example_model})"))
if candidate_stay != preferred_stay_id:
    display(Markdown(f"Preferred stay `{preferred_stay_id}` was not present in the local artifacts, so the notebook fell back to `{candidate_stay}`."))

for horizon_h in example_horizons:
    horizon_eval = evaluation_predictions[
        evaluation_predictions["model_name"].eq(example_model)
        & evaluation_predictions["horizon_h"].eq(horizon_h)
        & evaluation_predictions["stay_id_global"].eq(candidate_stay)
    ].copy()
    horizon_all_valid = all_valid_predictions[
        all_valid_predictions["model_name"].eq(example_model)
        & all_valid_predictions["horizon_h"].eq(horizon_h)
        & all_valid_predictions["stay_id_global"].eq(candidate_stay)
    ].copy()
    if horizon_all_valid.empty:
        display(Markdown(f"#### Horizon {horizon_h}h: stay not available"))
        continue
    horizon_all_valid["present_in_predictions_csv"] = horizon_all_valid["instance_id"].isin(horizon_eval["instance_id"])
    summary = pd.DataFrame(
        [
            {
                "horizon_h": horizon_h,
                "evaluation_rows": int(horizon_eval.shape[0]),
                "all_valid_rows": int(horizon_all_valid.shape[0]),
                "additional_scored_unlabeled_rows": int((~horizon_all_valid["present_in_predictions_csv"]).sum()),
            }
        ]
    )
    display(Markdown(f"#### Horizon {horizon_h}h"))
    display(summary)
    display(
        horizon_all_valid[
            [
                "instance_id",
                "block_index",
                "prediction_time_h",
                "label_value",
                "is_labelable",
                "unlabeled_reason",
                "present_in_predictions_csv",
                "predicted_probability",
            ]
        ].sort_values(["prediction_time_h", "block_index"])
    )


## 2. Cohort-Level Trajectory Coverage Summary

For trajectory analysis we care about how many blocks become newly visible once we move from the evaluation-only subset to all valid scored instances. The summary below focuses on proxy non-survivors, defined conservatively from the exported table as stays with at least one positive labelable row or any `non_survivor` unlabeled reason.


In [ ]:
coverage_non_survivors = stay_coverage[stay_coverage["proxy_non_survivor"]].copy()
coverage_summary = coverage_non_survivors.groupby(["model_name", "horizon_h"], as_index=False).agg(
    non_survivor_stays=("stay_id_global", "nunique"),
    mean_scored_blocks=("scored_blocks", "mean"),
    median_scored_blocks=("scored_blocks", "median"),
    mean_evaluation_labeled_blocks=("evaluation_labeled_blocks", "mean"),
    median_evaluation_labeled_blocks=("evaluation_labeled_blocks", "median"),
    mean_additional_scored_blocks=("additional_scored_unlabeled_blocks", "mean"),
    median_additional_scored_blocks=("additional_scored_unlabeled_blocks", "median"),
    pct_with_any_additional_block=("additional_scored_unlabeled_blocks", lambda s: float(pd.Series(s).gt(0).mean())),
)
display(coverage_summary.round(2))

plot_df = coverage_non_survivors[
    coverage_non_survivors["model_name"].eq(PRIMARY_MODEL)
    & coverage_non_survivors["horizon_h"].isin(PRIMARY_HORIZONS)
].copy()
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
sns.boxplot(data=plot_df, x="horizon_h", y="scored_blocks", ax=axes[0], color="#6997bf")
axes[0].set_title("All-valid scored blocks per non-survivor stay")
axes[0].set_xlabel("Horizon (h)")
axes[0].set_ylabel("Scored blocks")
sns.boxplot(data=plot_df, x="horizon_h", y="additional_scored_unlabeled_blocks", ax=axes[1], color="#d09b5b")
axes[1].set_title("Additional scored-but-unlabeled blocks gained")
axes[1].set_xlabel("Horizon (h)")
axes[1].set_ylabel("Additional blocks")
plt.show()


## 3. Descriptive Trajectory Features For Non-Survivors

These per-stay features are deliberately simple and Chapter 1 friendly: first risk, last risk, maximum risk, mean risk, first-to-last change, a final-window slope, threshold coverage, whether the maximum occurs in the final block, and a basic upward-step fraction.


In [ ]:
non_survivor_features = trajectory_features[trajectory_features["proxy_non_survivor"]].copy()
feature_summary = non_survivor_features.groupby(["model_name", "horizon_h"], as_index=False).agg(
    stay_count=("stay_id_global", "nunique"),
    median_first_risk=("first_risk", "median"),
    median_last_risk=("last_risk", "median"),
    median_max_risk=("max_risk", "median"),
    median_change_first_to_last=("change_first_to_last", "median"),
    median_final_window_slope=("final_window_slope", "median"),
    pct_max_in_final_block=("max_occurs_in_final_block", lambda s: float(pd.Series(s).mean())),
    pct_all_blocks_ge_0_05=("frac_blocks_ge_0_05", lambda s: float(pd.Series(s).eq(1.0).mean())),
    pct_last_risk_lt_0_05=("last_risk", lambda s: float(pd.Series(s).lt(0.05).mean())),
)
display(feature_summary.round(3))

feature_table_for_review = non_survivor_features[
    non_survivor_features["model_name"].eq(PRIMARY_MODEL)
    & non_survivor_features["horizon_h"].isin(PRIMARY_HORIZONS)
].sort_values(["horizon_h", "last_risk"], ascending=[True, False])
display(feature_table_for_review.head(20).round(3))


## 4. Visual Summaries

Keep the figure set small: a few representative individual trajectories, an aligned view of risk in the final scored blocks, a compact distribution view for a few trajectory features, and an optional hard-case comparison if the Chapter 1 hard-case artifact is available.


In [ ]:
visual_model = PRIMARY_MODEL
visual_horizon = 8
visual_subset = non_survivor_features[
    non_survivor_features["model_name"].eq(visual_model)
    & non_survivor_features["horizon_h"].eq(visual_horizon)
].copy()

representative_ids = []
if not visual_subset.empty:
    representative_ids.extend(visual_subset.nlargest(1, "mean_risk")["stay_id_global"].tolist())
    representative_ids.extend(visual_subset.nlargest(1, "change_first_to_last")["stay_id_global"].tolist())
    representative_ids.extend(visual_subset.nsmallest(1, "last_risk")["stay_id_global"].tolist())
    late_peak_subset = visual_subset[visual_subset["max_occurs_in_final_block"]]
    if not late_peak_subset.empty:
        representative_ids.extend(late_peak_subset.nlargest(1, "max_risk")["stay_id_global"].tolist())
representative_ids = list(dict.fromkeys(representative_ids))[:4]

trajectory_subset = all_valid_predictions[
    all_valid_predictions["model_name"].eq(visual_model)
    & all_valid_predictions["horizon_h"].eq(visual_horizon)
    & all_valid_predictions["stay_id_global"].isin(representative_ids)
].copy()
trajectory_subset = trajectory_subset.sort_values(["stay_id_global", "prediction_time_h", "block_index"], kind="stable")

if representative_ids:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=False, sharey=True, constrained_layout=True)
    axes = axes.ravel()
    for ax, stay_id in zip(axes, representative_ids):
        stay_df = trajectory_subset[trajectory_subset["stay_id_global"].eq(stay_id)].copy()
        ax.plot(stay_df["prediction_time_h"], stay_df["predicted_probability"], marker="o", color="#235789")
        unlabeled = stay_df[~stay_df["is_labelable"].fillna(False)]
        if not unlabeled.empty:
            ax.scatter(unlabeled["prediction_time_h"], unlabeled["predicted_probability"], color="#c1292e", s=60, label="scored only")
        ax.set_title(str(stay_id))
        ax.set_xlabel("Prediction time (h)")
        ax.set_ylabel("Predicted mortality risk")
    for ax in axes[len(representative_ids):]:
        ax.axis("off")
    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper center", ncol=len(labels), frameon=False)
    plt.show()

aligned_plot_df = all_valid_predictions[
    all_valid_predictions["model_name"].eq(PRIMARY_MODEL)
    & all_valid_predictions["horizon_h"].isin(PRIMARY_HORIZONS)
].merge(
    stay_coverage[["model_name", "horizon_h", "stay_id_global", "proxy_non_survivor"]],
    on=["model_name", "horizon_h", "stay_id_global"],
    how="left",
)
aligned_plot_df = aligned_plot_df[aligned_plot_df["proxy_non_survivor"]].copy()
aligned_plot_df = aligned_plot_df.sort_values(["horizon_h", "stay_id_global", "prediction_time_h", "block_index"], kind="stable")
aligned_plot_df["block_order"] = aligned_plot_df.groupby(["horizon_h", "stay_id_global"]).cumcount()
stay_sizes = aligned_plot_df.groupby(["horizon_h", "stay_id_global"])["instance_id"].transform("size")
aligned_plot_df["blocks_from_final"] = aligned_plot_df["block_order"] - (stay_sizes - 1)
aligned_summary = aligned_plot_df[aligned_plot_df["blocks_from_final"].between(-6, 0)].groupby(["horizon_h", "blocks_from_final"], as_index=False).agg(
    median_risk=("predicted_probability", "median"),
    q25=("predicted_probability", lambda s: float(pd.Series(s).quantile(0.25))),
    q75=("predicted_probability", lambda s: float(pd.Series(s).quantile(0.75))),
)
fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
palette = {8: "#235789", 24: "#c1292e"}
for horizon_h, horizon_df in aligned_summary.groupby("horizon_h"):
    horizon_df = horizon_df.sort_values("blocks_from_final")
    ax.plot(horizon_df["blocks_from_final"], horizon_df["median_risk"], marker="o", label=f"{horizon_h}h", color=palette.get(int(horizon_h), None))
    ax.fill_between(horizon_df["blocks_from_final"], horizon_df["q25"], horizon_df["q75"], alpha=0.15, color=palette.get(int(horizon_h), None))
ax.set_title("Median risk over the final scored blocks for proxy non-survivors")
ax.set_xlabel("Blocks from final scored block")
ax.set_ylabel("Predicted mortality risk")
ax.legend(frameon=False)
plt.show()

distribution_df = non_survivor_features[
    non_survivor_features["model_name"].eq(PRIMARY_MODEL)
    & non_survivor_features["horizon_h"].isin(PRIMARY_HORIZONS)
][["horizon_h", "first_risk", "last_risk", "change_first_to_last"]].melt(
    id_vars="horizon_h",
    var_name="trajectory_feature",
    value_name="value",
)
fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)
sns.boxplot(data=distribution_df, x="trajectory_feature", y="value", hue="horizon_h", ax=ax)
ax.set_title("Selected trajectory-feature distributions for proxy non-survivors")
ax.set_xlabel("")
ax.set_ylabel("Value")
plt.show()

hard_case_flags = load_hard_case_flags(PRIMARY_MODEL)
if hard_case_flags is not None and "hard_case_flag" in hard_case_flags.columns:
    hard_case_plot_df = non_survivor_features.merge(
        hard_case_flags[["stay_id_global", "horizon_h", "hard_case_flag"]].drop_duplicates(),
        on=["stay_id_global", "horizon_h"],
        how="left",
    )
    hard_case_plot_df = hard_case_plot_df[hard_case_plot_df["horizon_h"].isin(PRIMARY_HORIZONS)].copy()
    fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
    sns.boxplot(data=hard_case_plot_df, x="hard_case_flag", y="last_risk", hue="horizon_h", ax=ax)
    ax.set_title("Last-risk distribution for hard-case vs other fatal stays")
    ax.set_xlabel("Hard-case flag")
    ax.set_ylabel("Last predicted risk")
    plt.show()
else:
    display(Markdown("Hard-case stay-level flags were not found, so the optional comparison plot was skipped."))


## 5. Short Interpretation Notes

These notes are generated from the current artifact snapshot and are meant to stay tightly scoped to Chapter 1 interpretation rather than drift into open-ended exploration.


In [ ]:
def build_interpretation_markdown(feature_df: pd.DataFrame, model_name: str, horizons: tuple[int, ...]) -> Markdown:
    lines = [f"### {model_name.replace('_', ' ').title()} trajectory takeaways"]
    for horizon_h in horizons:
        subset = feature_df[
            feature_df["model_name"].eq(model_name)
            & feature_df["horizon_h"].eq(horizon_h)
        ].copy()
        if subset.empty:
            lines.append(f"- Horizon {horizon_h}h: no proxy non-survivor trajectories were available in the saved exports.")
            continue
        median_first = subset["first_risk"].median()
        median_last = subset["last_risk"].median()
        pct_late_peak = subset["max_occurs_in_final_block"].mean() * 100.0
        pct_low_last = subset["last_risk"].lt(0.05).mean() * 100.0
        pct_all_high = subset["frac_blocks_ge_0_10"].eq(1.0).mean() * 100.0
        median_change = subset["change_first_to_last"].median()
        lines.append(
            "- Horizon {h}h: median first risk {first:.3f}, median last risk {last:.3f}, "
            "median first-to-last change {change:.3f}, {late_peak:.1f}% of stays peak in the final block, "
            "{low_last:.1f}% still end below 0.05 risk, and {all_high:.1f}% stay above 0.10 throughout all scored blocks.".format(
                h=horizon_h,
                first=median_first,
                last=median_last,
                change=median_change,
                late_peak=pct_late_peak,
                low_last=pct_low_last,
                all_high=pct_all_high,
            )
        )
    lines.append(
        "- Read these numbers as trajectory descriptors, not formal performance metrics. The formal baseline metrics remain anchored to the labeled `predictions.csv` subset."
    )
    return Markdown("\n".join(lines))


display(build_interpretation_markdown(non_survivor_features, PRIMARY_MODEL, PRIMARY_HORIZONS))
